# Mini-Challenge 2.3: Segmentation & Morphological Operations

## Day 11: Data & Domain

### Use Case: Alpine Ibex Population Census via Automated Object Extraction

Alpine ibex conservation depends on accurate population counts and individual tracking. Conservation rangers and biologists in the Swiss National Park analyze field photographs and camera trap imagery to:
1. **Detect and locate individual ibex** within complex natural scenes (sky, rock, vegetation)
2. **Measure individual properties** (size, shape) to support species verification and age estimation
3. **Automate census workflows** to process hundreds of images rapidly vs manual frame-by-frame review

Reliable object segmentation is essential: distinguishing foreground (ibex) from background (rocks, sky) despite variable lighting, occlusion, and complex texture.

### Problem Statement

**Challenge**: Single images may contain multiple ibex (2 in our reference image) with vastly different backgrounds (sky vs rock). Simple thresholding often fails because:
- Alpine ibex fur exhibits variable intensity (shadows, highlights, texture)
- Backgrounds (rock, sky) may have overlapping intensity ranges with fur
- Over-sensitive segmentation fragments objects; under-sensitive segmentation merges them

**Approach**: Combine intensity-based thresholding with morphological operations (erosion, dilation) to refine raw segmentation while preserving object shapes and extracting meaningful properties (area, perimeter, skeleton centerlines).

### Objective

Investigate how threshold selection and morphological operations affect object extraction accuracy and property stability. The goal: determine optimal parameter ranges for reliable automated counting and measurement in alpine ibex surveys.

## Day 12: Methodological Design

### Segmentation Strategy

**Thresholding Method**: Global thresholding using Otsu's automatic threshold selection, plus manual threshold adjustments for comparison.
- **Otsu's method**: Automatically selects threshold that maximizes between-class variance (reproducible, no manual tuning)
- **Manual adjustment**: Allows exploration of suboptimal thresholds to study robustness

**Morphological Operations**: Sequences of erosion and dilation to refine binary masks
- **Opening** (erosion → dilation): Removes small noise (salt-and-pepper artifacts) while preserving larger objects
- **Closing** (dilation → erosion): Fills small internal holes, connects fragmented regions
- **Kernel size**: Larger kernels → more aggressive processing; smaller kernels → minimal change

### Object Property Measures

For each extracted object:
1. **Area** (pixel count): Object size; supports counting and filtering by size
2. **Perimeter** (boundary length): Shape complexity; elongated objects vs compact
3. **Skeleton** (morphological skeleton): Centerline representation; aids shape recognition and pose estimation

### Configuration Definitions

| Configuration | Threshold Method | Kernel Size | Morphology | Purpose |
|:---|:---|---:|:---|:---:|
| **C0 Baseline** | Otsu automatic | 3×3 | Opening only | Minimal processing, automatic threshold |
| **C1 Refined** | Otsu + manual adjust | 5×5 | Opening + closing | Cleaner objects, remove noise and fill holes |
| **C2 Aggressive** | Manual lower threshold | 7×7 | Opening + closing | Capture more detail, may include boundary noise |

**Parameter Justification**:
- Otsu is reproducible, automatic, and widely used in wildlife detection systems
- Kernel sizes scale from minimal (3×3) to aggressive (7×7) morphology
- Opening removes salt-and-pepper noise typical in alpine photography
- Closing fills internal voids in dark fur regions, improving connectivity
- Manual threshold in C2 set 20% below Otsu value to test sensitivity to under-segmentation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import cv2
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully
